# Document Parsing using DeepSeek-OCR and OpenVINO

DeepSeek-OCR, a VLM designed as a preliminary proof-of-concept for efficient vision-text compression.  DeepSeek-OCR consists of two components: DeepEncoder and DeepSeek3B-MoE-A570M as the decoder. Specifically, DeepEncoder serves as the core engine, designed to maintain low activations under high-resolution input while achieving high compression ratios to ensure an optimal and manageable number of vision tokens.

<img width="1882" height="566" alt="image" src="https://github.com/user-attachments/assets/6965428f-c4ec-4b44-be5a-4e35aa384122" />

More details can be found in the [paper](https://arxiv.org/pdf/2510.18234), original [repository](https://github.com/deepseek-ai/DeepSeek-OCR) and [model card](https://huggingface.co/deepseek-ai/DeepSeek-OCR).

In this tutorial we consider how to convert and run DeepSeek-OCR models using [OpenVINO](https://github.com/openvinotoolkit/openvino) and optimize it using [NNCF](https://github.com/openvinotoolkit/nncf).


#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert model to OpenVINO Intermediate Representation](#Convert-model-to-OpenVINO-Intermediate-Representation)
    - [Compress model weights to 4-bit](#Compress-model-weights-to-4-bit)
    - [Prepare inference pipeline](#Prepare-inference-pipeline)
- [Run model inference](#Run-model-inference)
- [Interactive demo](#Interactive-demo)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).


<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/deepseek-ocr/deepseek-ocr.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)


In [1]:
import requests
from pathlib import Path

utility_files = ["cmd_helper.py", "notebook_utils.py", "pip_helper.py"]
base_utility_url = "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/"

for utility in utility_files:
    if not Path(utility).exists():
        r = requests.get(base_utility_url + utility)
        with open(utility, "w", encoding="utf-8") as f:
            f.write(r.text)

In [ ]:
from pip_helper import pip_install
import platform

pip_install("-Uq", "openvino>=2025.3", "nncf>=2.15")
pip_install(
    "-q",
    "torch==2.8.0",
    "transformers==4.46.3",
    "tokenizers==0.20.3",
    "torchvision==0.23.0",
    "einops",
    "addict",
    "easydict",
    "huggingface_hub",
    "accelerate>=0.26.0",
    "PyMuPDF",
    "gradio>=4.19,<6",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
)

if platform.system() == "Darwin":
    pip_install("numpy<2.0")

In [3]:
from huggingface_hub import snapshot_download

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("deepseek-ocr.ipynb")

# Download DeepSeek-OCR model from HuggingFace
model_id = "deepseek-ai/DeepSeek-OCR"
local_dir = "./DeepSeek_OCR"

print(f"Downloading {model_id} to {local_dir}...")
snapshot_download(repo_id=model_id, local_dir=local_dir, local_dir_use_symlinks=False)
print(f"Model downloaded successfully to {local_dir}")

/home2/ethan/intel/openvino_notebooks/openvino_venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:982: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 21 files:   0%|          | 0/21 [00:00<?, ?it/s]

deepencoder.py: 0.00B [00:00, ?B/s]

Model downloaded successfully to ./DeepSeek_OCR


## Convert model to OpenVINO Intermediate Representation
[back to top ⬆️](#Table-of-contents:)


DeepSeek-OCR is PyTorch model. OpenVINO supports PyTorch models via conversion to OpenVINO Intermediate Representation (IR). [OpenVINO model conversion API](https://docs.openvino.ai/2024/openvino-workflow/model-preparation.html#convert-a-model-with-python-convert-model) should be used for these purposes. `ov.convert_model` function accepts original PyTorch model instance and example input for tracing and returns `ov.Model` representing this model in OpenVINO framework. Converted model can be used for saving on disk using `ov.save_model` function or directly loading on device using `core.complie_model`. 

The script `ov_deepseek_ocr_helper.py` contains helper function for model conversion, please check its content if you interested in conversion details.

<details>
  <summary><b>Click here for more detailed explanation of conversion steps</b></summary>
DeepSeek-OCR is autoregressive transformer generative model, it means that each next model step depends from model output from previous step. The generation approach is based on the assumption that the probability distribution of a word sequence can be decomposed into the product of conditional next word distributions. In other words, model predicts the next token in the loop guided by previously generated tokens until the stop-condition will be not reached (generated sequence of maximum length or end of string token obtained). The way the next token will be selected over predicted probabilities is driven by the selected decoding methodology. You can find more information about the most popular decoding methods in this <a href="https://huggingface.co/blog/how-to-generate">blog</a>. The entry point for the generation process for models from the Hugging Face Transformers library is the `generate` method. You can find more information about its parameters and configuration in the  <a href="https://huggingface.co/docs/transformers/v4.26.1/en/main_classes/text_generation#transformers.GenerationMixin.generate">documentation</a>. To preserve flexibility in the selection decoding methodology, we will convert only model inference for one step.

DeepSeek-OCR model consists of 3 parts:

* **Vision Model** for encoding input images into embedding space.
* **Embedding Model** for conversion input text tokens into embedding space.
* **Language Model** for generation answer based on input embeddings provided by Image Encoder and Input Embedding models.

</details>


### Compress model weights to 4-bit
[back to top ⬆️](#Table-of-contents:)


For reducing memory consumption, weights compression optimization can be applied using [NNCF](https://github.com/openvinotoolkit/nncf). 

<details>
    <summary><b>Click here for more details about weight compression</b></summary>
Weight compression aims to reduce the memory footprint of a model. It can also lead to significant performance improvement for large memory-bound models, such as Large Language Models (LLMs). LLMs and other models, which require extensive memory to store the weights during inference, can benefit from weight compression in the following ways:

* enabling the inference of exceptionally large models that cannot be accommodated in the memory of the device;

* improving the inference performance of the models by reducing the latency of the memory access when computing the operations with weights, for example, Linear layers.

[Neural Network Compression Framework (NNCF)](https://github.com/openvinotoolkit/nncf) provides 4-bit / 8-bit mixed weight quantization as a compression method primarily designed to optimize LLMs. The main difference between weights compression and full model quantization (post-training quantization) is that activations remain floating-point in the case of weights compression which leads to a better accuracy. Weight compression for LLMs provides a solid inference performance improvement which is on par with the performance of the full model quantization. In addition, weight compression is data-free and does not require a calibration dataset, making it easy to use.

`nncf.compress_weights` function can be used for performing weights compression. The function accepts an OpenVINO model and other compression parameters. Compared to INT8 compression, INT4 compression improves performance even more, but introduces a minor drop in prediction quality.

More details about weights compression, can be found in [OpenVINO documentation](https://docs.openvino.ai/2024/openvino-workflow/model-optimization-guide/weight-compression.html).
</details>

In [4]:
from pathlib import Path
import shutil

# Copy deepencoder.py to replace DeepSeek-OCR/deepencoder.py
source_file = Path("deepencoder.py")
target_file = Path("DeepSeek_OCR/deepencoder.py")

if source_file.exists():
    shutil.copy(source_file, target_file)
    print(f"Replaced {target_file} with {source_file}")
else:
    print(f"Source file {source_file} not found")

Replaced DeepSeek_OCR/deepencoder.py with deepencoder.py


In [5]:
import ipywidgets as widgets

to_compress = widgets.Checkbox(value=True, description="Compression")
to_compress

Checkbox(value=True, description='Compression')

In [6]:
import nncf
from ov_deepseek_ocr_helper import convert_deepseek_ocr

quantization_config = None

if to_compress.value:
    quantization_config = {
        "vision": {"mode": nncf.CompressWeightsMode.INT8_ASYM},
        "llm": {"mode": nncf.CompressWeightsMode.INT4_SYM, "group_size": 64, "ratio": 1.0},
    }

model_path = Path(local_dir) / ("FP16" if not to_compress.value else "INT4")

convert_deepseek_ocr(local_dir, model_path=model_path, quantization_config=quantization_config)

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cpu for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


✅ ./DeepSeek_OCR model already converted. You can find results in DeepSeek_OCR/INT4


PosixPath('DeepSeek_OCR/INT4')

### Prepare inference pipeline
[back to top ⬆️](#Table-of-contents:)


`OVDeepseekOCRForCausalLM` class defined in `ov_deepseek_ocr_helper.py` represents model inference class. It accepts path to model directory and target device for inference, like original pipeline, `OVDeepseekOCRForCausalLM` has `infer` method for getting answers. Besides that, it is compatible with original model processor class for preparing input.

In [7]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])

device

Dropdown(description='Device:', options=('CPU', 'GPU', 'AUTO'), value='CPU')

In [ ]:
from transformers import AutoTokenizer
from ov_deepseek_ocr_helper import OVDeepseekOCRForCausalLM

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = OVDeepseekOCRForCausalLM(model_path, device=device.value)

## Run model inference
[back to top ⬆️](#Table-of-contents:)


Let's check model prediction for document parsing. 

In [9]:
from PIL import Image

url = "https://huggingface.co/spaces/khang119966/DeepSeek-OCR-DEMO/resolve/main/doc_markdown.png"
file_name = "doc_markdown.png"
if not Path(file_name).exists():
    Image.open(requests.get(url, stream=True).raw).save(file_name)

In [10]:
prompt = "<image>\n<|grounding|>Convert the document to markdown. "
output_path = "."

res = model.infer(
    tokenizer,
    prompt=prompt,
    image_file=file_name,
    output_path=output_path,
    base_size=1024,
    image_size=640,
    crop_mode=True,
    save_results=True,
    test_compress=True,
)

/home2/ethan/intel/openvino_notebooks/openvino_venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|ref|>sub_title<|/ref|><|det|>[[124, 71, 625, 95]]<|/det|>
## Aspire OCR and Barcode Recognition  

<|ref|>text<|/ref|><|det|>[[124, 106, 862, 143]]<|/det|>
High performance, royalty- free OCR and barcode recognition on Windows, Linux, Mac OS and Unix.  

<|ref|>text<|/ref|><|det|>[[124, 153, 861, 247]]<|/det|>
Aspire OCR (optical character recognition) and barcode recognition SDK offers a high performance library for you to equip your Java applications (Java applets, web applications, Swing/JavaFX components, JEE enterprise applications), C#/VB.NET applications, and C/C++/Python applications with functionality of extracting text and barcode information from scanned documents.  

<|ref|>sub_title<|/ref|><|det|>[[124, 262, 485, 282]]<|/det|>
## Convert Images To Searchable PDF  

<|ref|>text<|/ref|><|det|>[[124, 291, 870, 327]]<|/det|>
With a few lines of code, you can convert various formats of images such as JPEG, PNG, and TIFF into searchable PDF.  

<|ref|>table<|/ref|><|det|>[[124

image: 0it [00:00, ?it/s]
other: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:00<00:00, 166111.05it/s]


## Interactive demo
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from gradio_helper import make_demo

demo = make_demo(model, tokenizer)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/